In [1]:
import json
import pandas as pd
from elsapy.elsclient import ElsClient
from elsapy.elssearch import ElsSearch
from elsapy.elsprofile import ElsAuthor, ElsAffil
from elsapy.elsdoc import FullDoc, AbsDoc
import numpy as np
from tqdm import tqdm
import networkx as nx
#from community import community_louvain
import matplotlib.pyplot as plt
import matplotlib.colors
import matplotlib.colors as mcolors

ModuleNotFoundError: No module named 'elsapy'

In [ ]:
# obtain a scopus API key from elsevier developer bportal
apikey='66467a6f955692223b0c98d1f14675442d

client = ElsClient(apikey)

In [ ]:
# Function to fetch detailed information for a single document
def fetch_document_info(eid):
    #doc = FullDoc(scopus_id=eid)
    #doc = FullDoc(eid)
    doc = AbsDoc(scp_id=eid)
    if doc.read(client):
        coredata = doc.data['coredata']
        return {
            'scopus_id': eid,
            'title': coredata.get('dc:title'),
            'year': coredata.get('prism:coverDate', '')[:4],  # Extract the year
            'abstract': coredata.get('dc:description')
        }
    return None

In [ ]:
# searching in title, abstract or keyword using a query
query = "TITLE-ABS-KEY(social AND network) AND SUBJAREA(SOCI) AND PUBYEAR > 2020"

In [ ]:

# Initialize ElsSearch
print(query)
search = ElsSearch(query, 'scopus')
search.execute(client)

search.execute(client, get_all = True)
print ("search has", search.num_res, "out of", search.tot_num_res, "potential results")
print("Has all results?", search.hasAllResults())

print(f"Found {search.results_df.shape[0]} documents")
#searches.append(search)

documents_info = []
for eid in search.results_df['dc:identifier']:
    eid = eid.split(':')[-1]  # Extract the actual Scopus ID from the identifier
    #print('eid=',eid)
    doc_info = fetch_document_info(eid)
    #print('doc_info=',doc_info)
    if doc_info:
        documents_info.append(doc_info)
        print(f"Fetched details for {eid}")
    else:
        print(f"could not Fetch details for {eid}")

# Convert to a DataFrame for easier handling
df = pd.DataFrame(documents_info)

# Save the results to a CSV file
df.to_csv('scopus_search_documents_query_'+queries[i]+'.csv', index=False)

print("Document details saved to csv'")
